# DSPy: Programming Instead of Prompting

In this notebook, we'll implement a basic DSPy pipeline to demonstrate how to optimize prompts automatically using **Signatures** and **BootstrapFewShot**.

In [ ]:
# !pip install dspy-ai openai
# !pip install pai-llm-config

In [ ]:
from pai_llm_config import config

dspy = config.dspy_client("smart")

## 1. Basic QA

In [ ]:
qa = dspy.Predict("question -> answer")
result = qa(question="What is the capital of France?")
print(result)

dspy.settings.lm.inspect_history(n=1)

## 2. COT

In [ ]:
class MathSolver(dspy.Signature):
    """Solve a math problem step-by-step."""
    problem = dspy.InputField()
    reasoning = dspy.OutputField(desc="逐步推理过程")
    answer = dspy.OutputField(desc="最终答案")

solver = dspy.ChainOfThought(MathSolver)
result = solver(problem="What is 2 + 2?")
print(result.reasoning)
print(result.answer)

dspy.settings.lm.inspect_history(n=1)


## 3. Implements a SentimentClassifier

In [ ]:
class SentimentClassifier(dspy.Signature):
    """Classify the sentiment of a sentence as Positive, Negative, or Neutral."""
    sentence = dspy.InputField()
    sentiment = dspy.OutputField(desc="One of: Positive, Negative, Neutral")
    confidence = dspy.OutputField(desc="Confidence score (0-100)")

classifier = dspy.Predict(SentimentClassifier)
result = classifier(sentence="I f**k this app!")
print(result)

dspy.settings.lm.inspect_history(n=1)

## 4. Multi-Step Pipeline

In [ ]:
class Summarizer(dspy.Signature):
    """Summary the document"""
    document = dspy.InputField()
    summary = dspy.OutputField(desc="Summary of the document use 3 sentences")


class KeywordExtractor(dspy.Signature):
    """Extract keywords from the document"""
    summary = dspy.InputField()
    keywords = dspy.OutputField(desc="Keywords of the document, use 5 words and separated by comma")


class DocumentAnlaysis(dspy.Module):
    """Analyze the document"""
    def __init__(self):
        self.summarizer = dspy.Predict(Summarizer)
        self.keyword_extractor = dspy.Predict(KeywordExtractor)
    
    def forward(self, document):
        summary = self.summarizer(document=document).summary
        keywords = self.keyword_extractor(summary=summary).keywords
        return dspy.Prediction(summary=summary, keywords=keywords)

analyzer = DocumentAnlaysis()
long_document = (
    "This is a mock long document. "
    "It contains multiple sentences to simulate real document content for analysis. "
    "Sentiment classification, summarization, and keyword extraction are often performed on such substantial texts. "
    "Here we discuss various aspects of artificial intelligence and its impact on industry. "
    "Historically, AI has revolutionized many fields, leading to increased efficiency and new opportunities. "
    "However, ethical considerations and technical challenges remain a focus of ongoing research. "
    "In this document, we review the main achievements and current limitations, and provide insights into future trends."
)
result = analyzer(document=long_document)
print(result)

## 5. RAG

In [ ]:
class RAGSignature(dspy.Signature):
    context = dspy.InputField(desc="Contextual information about the topic")
    question = dspy.InputField(desc="Question about the context")
    answer = dspy.OutputField(desc="Answer to the question")

class RAGPipeline(dspy.Module):
    def __init__(self, retriever):
        self.retriever = retriever
        self.genearte = dspy.ChainOfThought(RAGSignature)

    def forward(self, question):
        docs = self.retriever.search(question)
        context = "\n".join([doc.page_content for doc in docs])
        return self.genearte(context=context, question=question)


# Implement a simple Retriever. This stores documents in a list and retrieves documents containing the keyword.
class SimpleRetriever:
    def __init__(self, documents):
        # documents: list of Document objects or dicts with 'page_content'
        self.documents = documents

    def search(self, query, top_k=3):
        # Simple implementation: Return documents containing the query string
        matched = []
        for doc in self.documents:
            text = getattr(doc, 'page_content', None)
            if text is None:
                text = getattr(doc, 'text', None)
            if text is None and isinstance(doc, dict):
                text = doc.get('page_content') or doc.get('text')
            if text and query.lower() in text.lower():
                matched.append(doc)
        return matched[:top_k] if matched else self.documents[:top_k]

# Define a simple Document structure for testing
class Document:
    def __init__(self, page_content):
        self.page_content = page_content

# Example documents
documents = [
    Document("The capital of France is Paris."),
    Document("2 + 2 equals 4."),
    Document("DSPy is a framework for neural programming."),
]

retriever = SimpleRetriever(documents)
pipeline = RAGPipeline(retriever)

result = pipeline(question="What is the DSPy?")
print(result)

## 6. ReAct Agent

In [ ]:
def search_web(query:str)->str:
    """Search web information"""
    return f"Query result: information of {query}..."

def calculate(expression: str)->str:
    """Calculate math expression"""
    return str(eval(expression))

class ResearchAgent(dspy.Module):

    def __init__(self):
        self.agent = dspy.ReAct(
            "question -> answer",
            tools=[search_web, calculate]
        )

    def forward(self, question):
        return self.agent(question=question)

agent = ResearchAgent()
result = agent(question="What is the total global GDP in 2024? What is it converted to RMB?")
print(result)

## 7. Setting up the Optimizer

The power of DSPy lies in automatically selecting the best Few-shot examples (Bootstrapping).

In [ ]:
from dspy.teleprompt import BootstrapFewShot

# 1. Example Training Data (Small Golden Set)
# Note: Use .with_inputs('sentence') to specify the input fields for optimization.
trainset = [
    dspy.Example(sentence="The steak at this restaurant was absolutely perfect!", sentiment="Positive", confidence=100).with_inputs('sentence'),
    dspy.Example(sentence="Waited two hours and the food was still cold. Terrible experience.", sentiment="Negative", confidence=100).with_inputs('sentence'),
    dspy.Example(sentence="The product is average, and the price is reasonable.", sentiment="Neutral", confidence=80).with_inputs('sentence'),
    dspy.Example(sentence="Shipping was fast, but the package was damaged.", sentiment="Negative", confidence=90).with_inputs('sentence'),
    dspy.Example(sentence="Customer service was great and solved my problem quickly.", sentiment="Positive", confidence=95).with_inputs('sentence')
]

# 2. Metric: Check if prediction is correct (exact match of sentiment label)
def validate_sentiment(example, pred, trace=None):
    return example.sentiment.lower() == pred.sentiment.lower()

# 3. Define the program we want to optimize.
# ChainOfThought allows the optimizer to record the reasoning steps.
# Pass in the class name SentimentClassifier; Predictor will instantiate it.
program_to_optimize = dspy.ChainOfThought(SentimentClassifier)

# 4. Set up the optimizer.
# max_bootstrapped_demos=3 means at most 3 successful training examples are used as in-context demos.
teleprompter = BootstrapFewShot(metric=validate_sentiment, max_bootstrapped_demos=3)

# 5. Compile — real optimization happens here.
# Runs the trainset, finds successful examples, and stores them in the prompt.
compiled_classifier = teleprompter.compile(program_to_optimize, trainset=trainset)

print("DSPy pipeline optimization complete.")

# 6. Test the optimized classifier.
test_sentence = "Although it's a bit expensive, it's really effective."
result = compiled_classifier(sentence=test_sentence)
print(f"Test input: {test_sentence}")
print(f"Predicted sentiment: {result.sentiment}")
print(f"Confidence: {result.confidence}")

# 7. Inspect the evolved prompt.
# You will see that the prompt now automatically includes a few 'successful examples' from the trainset and their reasoning process.
dspy.settings.lm.inspect_history(n=1)

In [ ]:
dspy.settings.lm.inspect_history(n=1)

## 8. Structured Output

DSPy works seamlessly with Pydantic for type-safe, structured extraction.

In [ ]:
from pydantic import BaseModel
from typing import List

class ProductReview(BaseModel):
    pros: List[str]
    cons: List[str]
    rating: int # 1-5
    recommendation: bool

class ReviewAnalyzer(dspy.Signature):
    """Analyze product reviews and provide structured output."""
    review_text = dspy.InputField()
    analysis: ProductReview = dspy.OutputField()

analyzer = dspy.TypedPredictor(ReviewAnalyzer)
result = analyzer(review_text="This phone has a great camera, but the battery life is poor. Overall, I give it a 3.")

print(f"Pros: {result.analysis.pros}")
print(f"Rating: {result.analysis.rating}")
print(f"Recommend: {result.analysis.recommendation}")

In [ ]:
class SearchQuery(dspy.Signature):
    """Generate search queries for a complex question."""
    question = dspy.InputField()
    context = dspy.InputField()
    query = dspy.OutputField()

class SimplifiedBaleen(dspy.Module):
    def __init__(self, retriever, max_hops=2):
        super().__init__()
        self.retriever = retriever
        self.generate_query = dspy.ChainOfThought(SearchQuery)
        self.generate_answer = dspy.ChainOfThought("context, question -> answer")
        self.max_hops = max_hops
    
    def forward(self, question):
        context = []
        for _ in range(self.max_hops):
            # 1. Generate new query based on current context
            query = self.generate_query(question=question, context=context).query
            # 2. Search for more information
            results = self.retriever.search(query)
            # 3. Add to context
            context.extend(results)
        
        # Finally, generate the answer
        return self.generate_answer(context=context, question=question)

# pipeline = SimplifiedBaleen(retriever)
# result = pipeline(question="How does DSPy compare to LangChain in terms of optimization?")

## 10. Multi-hop Reasoning (Baleen Architecture)

Multi-hop systems can solve complex questions by generating a series of search queries and gathering information step-by-step.

In [ ]:
import dspy

class SentenceCheck(dspy.Signature):
    text = dspy.InputField()
    is_friendly = dspy.OutputField(desc="boolean")

class FriendlyChat(dspy.Module):
    def __init__(self):
        super().__init__()
        self.check = dspy.Predict(SentenceCheck)
    
    def forward(self, text):
        res = self.check(text=text)
        # Runtime Assertion: If not friendly, DSPy triggers automatic retry logic
        dspy.Assert(res.is_friendly == 'True', "The output must maintain a friendly tone!")
        return res

# Assertions require enabling with_assertions()
chat = FriendlyChat().with_assertions(max_retries=3)

## 9. Assertions & Suggestions (DSPy 3.x)

Control your LLM pipeline with dynamic constraints to ensure quality. In DSPy 3.x, use `with_assertions()` to enable automatic retry logic.

## Why this is Industrial Standard:

1. **Model Agnostic**: Change the LM in `dspy.settings`, and the optimizer will find new best examples for THAT model.
2. **Metric Driven**: We optimize based on a function (`validate_sentiment`), not vibes.
3. **Modular**: We can stack modules like `ChainOfThought` inside `ProgramOfThought` effortlessly.